<a href="https://colab.research.google.com/github/kumisganteng/Data-Analytics-Projects/blob/energy_data_cleaning/Energy_Consumption_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Data Ingestion

##1.1 Pemuatan Data

In [1]:
import pandas as pd

AEP_hourly_energy_dataset = "AEP_hourly.csv"
energy_data = pd.read_csv(AEP_hourly_energy_dataset)

##1.2 Inspeksi Awal

In [2]:
energy_data.shape

(121273, 2)

In [3]:
energy_data.head()

,Datetime,AEP_MW
0,2004-12-31 01:00:00,13478.0
1,2004-12-31 02:00:00,12865.0
2,2004-12-31 03:00:00,12577.0
3,2004-12-31 04:00:00,12517.0
4,2004-12-31 05:00:00,12670.0


In [4]:
energy_data.tail()

,Datetime,AEP_MW
121268,2018-01-01 20:00:00,21089.0
121269,2018-01-01 21:00:00,20999.0
121270,2018-01-01 22:00:00,20820.0
121271,2018-01-01 23:00:00,20415.0
121272,2018-01-02 00:00:00,19993.0


In [5]:
energy_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  121273 non-null  object 
 1   AEP_MW    121273 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.9+ MB


In [6]:
energy_data.describe()

,AEP_MW
count,121273.000000
mean,15499.513717
std,2591.399065
min,9581.000000
25%,13630.000000
50%,15310.000000
75%,17200.000000
max,25695.000000


In [7]:
energy_data.isnull().sum()

,0
Datetime,0
AEP_MW,0


In [8]:
energy_data.duplicated().sum()

np.int64(0)

###Cek Rentang Waktu

In [9]:
# cek rentang waktu data
print(energy_data['Datetime'].min())
print(energy_data['Datetime'].max())

# cek missing hours
energy_data['Datetime'] = pd.to_datetime(energy_data['Datetime'])
expected = pd.date_range(start=energy_data['Datetime'].min(),
                         end=energy_data['Datetime'].max(),
                         freq='h')
missing_hours = expected.difference(energy_data['Datetime'])
print(f"Missing hours: {len(missing_hours)}")

# cek duplikasi
print(f"Duplikasi: {energy_data.duplicated().sum()}")

# cek missing values
print(energy_data.isnull().sum())

2004-10-01 01:00:00
2018-08-03 00:00:00
Missing hours: 27
Duplikasi: 0
Datetime    0
AEP_MW      0
dtype: int64


#2. Data Cleaning

##2.1 Handle Missing Hours dan Duplikasi

In [17]:
print(energy_data.index.duplicated().sum())

4


In [19]:
# menghapus duplikasi di index
energy_data = energy_data[~energy_data.index.duplicated(keep='first')]

# mengcek setelah hapus duplikat
print(f"Duplikat setelah dibersihkan: {energy_data.index.duplicated().sum()}")
print(f"Shape: {energy_data.shape}")

# reindex
energy_data = energy_data.reindex(expected)
print(f"Missing values setelah reindex: {energy_data.isnull().sum().sum()}")

# interpolasi
energy_data['AEP_MW'] = energy_data['AEP_MW'].interpolate(method='time')
print(f"Missing values setelah interpolasi: {energy_data.isnull().sum().sum()}")
print(f"Shape akhir: {energy_data.shape}")

Duplikat setelah dibersihkan: 0
Shape: (121269, 1)
Missing values setelah reindex: 27
Missing values setelah interpolasi: 0
Shape akhir: (121296, 1)
Duplikat setelah dibersihkan: 0
Shape: (121296, 1)
Missing values setelah reindex: 0
Missing values setelah interpolasi: 0
Shape akhir: (121296, 1)


### Insight yang didapat
Pada awalnya ketika belum set index untuk Datetime duplikasi masih 0,
Namun saat setelah saya lakukan set index untuk Datetime, ternyata ditemukan 4 duplikasi. Awalnya saya bingung karena waktu inspeksi awal
duplikasi 0, tapi ternyata ini bukan duplikasi biasa.

Setelah ditelusuri, ternyata ini terjadi karena **Daylight Saving Time (DST)**
yaitu sebuah kebijakan di Amerika dimana jam diputar mundur 1 jam di musim gugur.
Jadi ada jam yang sama muncul 2 kali di data, misalnya jam 02:00 bisa
tercatat dua kali dalam satu hari. Ini hal baru yang saya pelajari
dari dataset ini.

Penanganan yang dilakukan:
1. Hapus duplikat — keep baris pertama, buang yang kedua
2. Reindex dengan expected datetime (semua jam yang seharusnya ada)
3. Interpolasi untuk mengisi 27 jam yang kelewat

Hasilnya shape bertambah dari 121.269 ke 121.296 setelah
27 jam baru ditambahkan dan diisi dengan interpolasi.

##2.2 Validasi Range AEP_MW


In [20]:
# validasi range AEP_MW
print(f"Min AEP_MW: {energy_data['AEP_MW'].min()}")
print(f"Max AEP_MW: {energy_data['AEP_MW'].max()}")
print(f"Nilai negatif: {(energy_data['AEP_MW'] < 0).sum()}")
print(f"Nilai 0: {(energy_data['AEP_MW'] == 0).sum()}")

Min AEP_MW: 9581.0
Max AEP_MW: 25695.0
Nilai negatif: 0
Nilai 0: 0


##2.3 Validasi Tipe Data

In [21]:
# validasi tipe data
print(energy_data.dtypes)
print(f"\nTipe index: {energy_data.index.dtype}")

AEP_MW    float64
dtype: object

Tipe index: datetime64[ns]


### Data Cleaning (Summary)

Dataset ini tergolong sangat bersih sejak awal, jadi proses
cleaning tidak terlalu banyak yang perlu dilakukan.

Yang ditemukan dan ditangani:

**Duplikasi di index Datetime (4 baris)**
Awalnya waktu inspeksi awal duplikasi 0, tapi setelah Datetime
dijadikan index ternyata ada 4 duplikat. Setelah ditelusuri,
ini terjadi karena Daylight Saving Time (DST) di Amerika —
kebijakan dimana jam diputar mundur 1 jam di musim gugur,
sehingga ada jam yang sama tercatat dua kali. Ini hal baru
yang saya pelajari dari dataset ini. Duplikat dihapus dengan
keep='first'.

**Missing hours (27 jam)**
Ada 27 jam yang tidak tercatat dari 14 tahun data — jumlah
yang sangat kecil dan wajar. Ditangani dengan reindex untuk
menambahkan baris kosong, lalu diisi dengan interpolasi
method='time' karena data time series nilainya berkesinambungan.

Setelah cleaning, shape data menjadi (121.296, 1) dengan
tidak ada missing values dan semua tipe data sudah sesuai.